In [1]:
import duckdb
import pandas as pd
import os
from datetime import datetime
from zoneinfo import ZoneInfo

In [2]:
con = duckdb.connect()

In [5]:
#agrouper pour jour et heur de execution en utilisant regexp_extract

con.execute("""
CREATE OR REPLACE VIEW delays_stop_snapshot AS
SELECT *,
       STRPTIME(
           regexp_extract(filename, '([0-9]{8}_[0-9]{6})'),
           '%Y%m%d_%H%M%S'
       ) AS snapshot_ts_temps
FROM read_csv_auto(
    './exports/avg_delay_by_stop_2025091[2-9]*.csv',
    HEADER=TRUE,
    AUTO_DETECT=TRUE,
    union_by_name=TRUE,
    SAMPLE_SIZE=-1,
    FILENAME=TRUE
);
""")

In [11]:
df = con.execute("""
SELECT 
    CAST(stop_id AS VARCHAR) AS stop_id,
    stop_name,
    stop_lat,
    stop_lon,
    snapshot_ts_temps,
    avg_delay_min
FROM delays_stop_snapshot
ORDER BY stop_id, snapshot_ts_temps;
""").df()

df


,stop_id,stop_name,stop_lat,stop_lon,snapshot_ts_temps,avg_delay_min
0,1,Abattoirs,43.717886,7.284703,2025-09-12 10:51:20,NaN
1,1,Abattoirs,43.717886,7.284703,2025-09-12 13:19:13,-3.05
2,1,Abattoirs,43.717886,7.284703,2025-09-12 17:22:59,-3.01
3,1,Abattoirs,43.717886,7.284703,2025-09-12 18:02:53,-6.22
4,1,Abattoirs,43.717886,7.284703,2025-09-13 12:41:25,-0.40
...,...,...,...,...,...,...
85175,place_WSTIS1,Saint-Isidore,43.710274,7.194760,2025-09-17 18:03:28,NaN
85176,place_WSTIS1,Saint-Isidore,43.710274,7.194760,2025-09-18 12:15:21,NaN
85177,place_WSTIS1,Saint-Isidore,43.710274,7.194760,2025-09-19 12:44:28,NaN
85178,place_WSTIS1,Saint-Isidore,43.710274,7.194760,2025-09-19 12:49:12,NaN


In [12]:
# exploration pour trouver arrets critiques
df_summary = con.execute("""
SELECT 
    stop_id,
    stop_name,
    MIN(avg_delay_min) AS min_delay,
    MAX(avg_delay_min) AS max_delay
FROM delays_stop_snapshot
WHERE avg_delay_min IS NOT NULL
GROUP BY stop_id, stop_name
ORDER BY max_delay DESC;
""").df()

df_summary

,stop_id,stop_name,min_delay,max_delay
0,21265,RUE 18bis / Avenue 1,-0.28,35.13
1,8064,Horizon,-6.14,23.30
2,8077,La Candellera,-6.61,22.55
3,1183,Saint-Albert,-28.03,22.45
4,619,La Pignata,-13.57,22.15
...,...,...,...,...
2401,4301,Jade,-4.00,-4.00
2402,947,Avenue Mont Alban,-4.83,-4.83
2403,2685,Vauban,-17.52,-6.53
2404,5247,Centre Commercial CAP 3000,-10.52,-7.00


In [13]:
# a partir de l"exploration, filtre para arret visualisés horizon 21625, la candellera 8077 etc


df_ok = con.execute("""
    SELECT 
        CAST(stop_id AS VARCHAR) AS stop_id,
        stop_name,
        stop_lat,
        stop_lon,
        snapshot_ts_temps,
        avg_delay_min
    FROM delays_stop_snapshot
    WHERE stop_id IN ('21265','8064','8077','1183','619')
    ORDER BY stop_id, snapshot_ts_temps
""").df()

df_ok

,stop_id,stop_name,stop_lat,stop_lon,snapshot_ts_temps,avg_delay_min
0,1183,Saint-Albert,43.724773,7.295985,2025-09-12 10:51:20,NaN
1,1183,Saint-Albert,43.724773,7.295985,2025-09-12 13:19:13,22.45
2,1183,Saint-Albert,43.724773,7.295985,2025-09-12 17:22:59,-0.63
3,1183,Saint-Albert,43.724773,7.295985,2025-09-12 18:02:53,-0.02
4,1183,Saint-Albert,43.724773,7.295985,2025-09-13 12:41:25,NaN
...,...,...,...,...,...,...
90,8077,La Candellera,43.715188,7.313703,2025-09-17 18:03:28,0.83
91,8077,La Candellera,43.715188,7.313703,2025-09-18 12:15:21,NaN
92,8077,La Candellera,43.715188,7.313703,2025-09-19 12:44:28,1.39
93,8077,La Candellera,43.715188,7.313703,2025-09-19 12:49:12,0.00


In [14]:
EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def current_timestamp_string():
    return datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M%S")

csv_path = f"{EXPORT_DIR}/evolutiuon_delay_stop_Q7_{current_timestamp_string()}.csv"
df.to_csv(csv_path, index=False)
csv_path

'./exports/evolutiuon_delay_stop_Q7_20250922_153333.csv'